# ART SLIDE：シカゴ美術館から新規200作品を追加

こちらは**次回専用の別ノートブック**です。現在実行している旧ノートブックには変更を加えません。

このノートブックは、シカゴ美術館（Art Institute of Chicago）の公式Open Access API **のみ**を使用し、`ryouy/slides` に未登録の作品を**最大200点**収集します。合計200点ではなく、実行時点の既存作品に追加する**新規200点**が目標です。

## 重複ダウンロード防止

- 最初にGitHubの`artworks.ts`と`additional-artworks.ts`、既存画像一覧を読み込みます。すでに収集済みの作品は**画像ダウンロードの前**に除外します。
- 美術館の固有作品ID・作品URL・作者＋タイトル・IIIF画像ID・GitHub上の画像ファイル名を照合します。
- 検索語を跨いだ重複も除外します。
- **GitHubへ保存する直前にもう一度最新のデータを読み直し**、先に実行中のノートブックが保存した作品も除外します。
- 途中でmainが更新された場合は、競合を検知して最新データから統合し直します（無理な強制プッシュはしません）。

## 実行手順

1. Colabの左端の鍵アイコン（Secrets）で`GITHUB_TOKEN`を登録し、ノートブックへのアクセスを許可。トークンには`ryouy/slides`のContents「Read and write」権限が必要です。
2. 「ランタイム」→「すべてのセルを実行」。
3. 最後に表示されるGitHubコミットURLで、追加した件数と画像を確認してください。

保存先：`public/artworks/`（WebP）、`src/games/art-slide/data/additional-artworks.ts`、`src/games/art-slide/data/artworks-research.json`。これらは1コミットでまとめて更新します。既存`artworks.ts`・日替わり用固定リストは変更しません。

API側の条件や画像の有無によって、200点に満たない場合があります。画像はシカゴ美術館APIの`is_public_domain = true`を確認。美術館提供の詳しい解説文はCC BY 4.0として出典・加工情報を記録し、公開ページにも表示されます。

In [ ]:
%pip -q install requests pillow beautifulsoup4


In [ ]:
# ART SLIDE — 200 NEW artworks from the Art Institute of Chicago (AIC).
# This is a separate notebook; your previous Met+AIC Colab run is untouched.
TARGET_AIC = 200
AIC_PAGES_PER_QUERY = 8    # <= 10 with limit=80: API search limit is 1,000 per query
AIC_RESULTS_PER_PAGE = 80
AIC_PER_QUERY_QUOTA = 22  # diversity: do not fill the gallery from a single search
AIC_SEARCH_TERMS = [
    "Impressionism", "Post-Impressionism", "landscape", "portrait", "still life",
    "watercolor", "sea", "Japanese woodblock", "flowers", "city street", "river",
    "Paris", "gardens", "mountains", "women", "children", "rural life", "interior",
    "seascape", "Monet", "Renoir", "Degas", "Pissarro", "Cassatt", "Turner",
    "Cézanne", "Van Gogh", "Hokusai", "Manet", "painting", "prints", "nature",
]
MIN_YEAR = 1300
MAX_YEAR = 1925
GAME_EDGE = 1200
THUMB_EDGE = 420
MIN_IMAGE_EDGE = 450
MAX_DESCRIPTION_CHARS = 2500
DOWNLOAD_ZIP_BACKUP = False    # GitHub is the primary destination.
REPO = "ryouy/slides"
BRANCH = "main"

import base64, html, io, json, random, re, time, unicodedata, zipfile
from datetime import date
from urllib.parse import urlparse
import requests
from bs4 import BeautifulSoup
from PIL import Image, ImageOps
from google.colab import userdata, files

API = "https://api.github.com/repos/" + REPO
AIC_API = "https://api.artic.edu/api/v1"
CC0 = "https://creativecommons.org/publicdomain/zero/1.0/"
CC_BY = "https://creativecommons.org/licenses/by/4.0/"
session = requests.Session()
session.headers.update({"User-Agent": "ART-SLIDE-Colab/1.0 (sequential requests)",
                        "AIC-User-Agent": "ART-SLIDE-Colab/1.0"})

def json_api(url, params=None, pause=.65):
    for attempt in range(5):
        time.sleep(pause)
        try:
            r = session.get(url, params=params, timeout=(15, 60))
            if r.status_code in (429, 500, 502, 503, 504):
                wait = min(50, 3 * 2 ** attempt)
                print("Museum API:", r.status_code, "retrying in", wait, "seconds")
                time.sleep(wait)
                continue
            r.raise_for_status()
            return r.json()
        except requests.RequestException:
            if attempt == 4: raise
            time.sleep(min(40, 3 * 2 ** attempt))
    raise RuntimeError("Museum API unavailable: " + url)

def plain_text(value):
    soup = BeautifulSoup(html.unescape(str(value or "")), "html.parser")
    for tag in soup(["script", "style"]): tag.decompose()
    return re.sub(r"\s+", " ", soup.get_text(" ", strip=True)).strip()

def normalized(value):
    value = unicodedata.normalize("NFKD", str(value or ""))
    return re.sub(r"[^a-z0-9]+", " ", "".join(c for c in value if not unicodedata.combining(c)).lower()).strip()

def year_int(value):
    m = re.search(r"(?<!\d)(1[3-9]\d\d|20[0-2]\d)(?!\d)", str(value or ""))
    return int(m.group(1)) if m else None

def era(y):
    return "Before 1700" if y < 1700 else "1700s" if y < 1800 else "1800–1849" if y < 1850 else "1850–1879" if y < 1880 else "1880–1899" if y < 1900 else "1900s"

def theme(title, medium="", subjects="", japan=False):
    if japan: return "Japanese Prints"
    s = normalized(title + " " + medium + " " + subjects)
    for label, pattern in [
        ("Myth & Religion", r"saint|christ|madonna|angel|religio|myth|holy"),
        ("Still Life & Flowers", r"still life|flower|rose|vase|fruit|bouquet"),
        ("Portrait", r"portrait|self portrait|madame|lady|girl|woman|man"),
        ("Sea & Water", r"sea|river|water|ocean|beach|coast|wave|harbor|port|boat"),
        ("City & Street", r"city|town|street|avenue|bridge|piazza|paris"),
        ("Landscape", r"landscape|field|forest|mountain|tree|garden|hill|valley"),
    ]:
        if re.search(pattern, s): return label
    return "People & Daily Life"

def title_key(artist, title):
    return (normalized(artist), normalized(title))

def summary(title, artist, year, medium, other="", tags=None):
    bits = [title + " is a work by " + artist + "."]
    if year: bits.append("The museum dates this work to " + year + ".")
    if medium: bits.append("Its medium is " + medium + ".")
    if other: bits.append("The catalogue associates it with " + other + ".")
    if tags: bits.append("Its recorded subjects include " + ", ".join(tags[:5]) + ".")
    return " ".join(bits)

def download_webps(url):
    host = urlparse(url).hostname
    if host not in ("www.artic.edu", "images.metmuseum.org"):
        raise ValueError("Unapproved image host")
    time.sleep(.4)
    with session.get(url, stream=True, timeout=(15, 90)) as resp:
        resp.raise_for_status()
        parts, total = [], 0
        for chunk in resp.iter_content(65536):
            total += len(chunk)
            if total > 30 * 1024 * 1024: raise ValueError("Image too large")
            parts.append(chunk)
    im = ImageOps.exif_transpose(Image.open(io.BytesIO(b"".join(parts)))).convert("RGB")
    if min(im.size) < MIN_IMAGE_EDGE or max(im.size) / min(im.size) > 3.8:
        raise ValueError("Image unsuitable for slider")
    def webp(edge, quality):
        x = im.copy()
        x.thumbnail((edge, edge), Image.Resampling.LANCZOS)
        out = io.BytesIO()
        x.save(out, "WEBP", quality=quality, method=5)
        return out.getvalue(), x.size
    full, dimensions = webp(GAME_EDGE, 82)
    thumb, _ = webp(THUMB_EDGE, 76)
    return full, thumb, dimensions

def record(work_id, title, artist, year, start, end, medium, museum, source, image_url, dims, kind, description, desc_source=None):
    return {
        "id": work_id, "title": title, "artist": artist,
        "artistSort": normalized(artist).split(" ")[-1] + " " + normalized(artist),
        "year": year, "yearStart": start, "yearEnd": end,
        "era": era(start), "theme": kind, "medium": medium,
        "imageUrl": "/artworks/" + work_id + ".webp",
        "thumbnailUrl": "/artworks/" + work_id + "-thumb.webp",
        "width": dims[0], "height": dims[1], "museum": museum,
        "sourceUrl": source, "license": "CC0 1.0 (public-domain image)",
        "licenseUrl": CC0,
        "licenseBasis": museum + ": public-domain status verified via official API",
        "sourceImageUrl": image_url, "verifiedAt": date.today().isoformat(),
        "description": description,
        "descriptionKind": "museum_text" if desc_source else "metadata_summary",
        **({
          "descriptionAttribution": "Art Institute of Chicago",
          "descriptionSourceUrl": desc_source, "descriptionLicense": "CC BY 4.0",
          "descriptionLicenseUrl": CC_BY,
          "descriptionModifications": "Original museum HTML converted to plain text; whitespace normalized."
        } if desc_source else {}),
    }

token = (userdata.get("GITHUB_TOKEN") or "").strip()
if not token:
    raise RuntimeError("Colab Secrets (key icon): create GITHUB_TOKEN and enable notebook access.")
github = requests.Session()
github.headers.update({
    "Authorization": "Bearer " + token,
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
    "User-Agent": "ART-SLIDE-Colab/1.0",
})

def gh(method, path, payload=None):
    for attempt in range(5):
        r = github.request(method, API + path, json=payload, timeout=(15, 150))
        if r.status_code in (429, 500, 502, 503, 504) or (r.status_code == 403 and r.headers.get("Retry-After")):
            if attempt == 4: raise RuntimeError("GitHub temporarily unavailable")
            time.sleep(int(r.headers.get("Retry-After") or min(60, 3 * 2 ** attempt)))
            continue
        if not r.ok:
            try: detail = r.json().get("message", "")
            except ValueError: detail = ""
            raise RuntimeError("GitHub " + method + " " + path + ": HTTP " + str(r.status_code) + " " + detail[:180])
        return r.json()
    raise RuntimeError("GitHub API error")

def repo_text(path, optional=False, revision=None):
    r = github.get(API + "/contents/" + path, params={"ref": revision or BRANCH}, timeout=(15, 50))
    if optional and r.status_code == 404: return ""
    r.raise_for_status()
    data = r.json()
    return base64.b64decode(data["content"]).decode("utf-8")

def parse_list(path, export_name, optional=False, revision=None):
    text = repo_text(path, optional, revision)
    if not text and optional: return []
    match = re.search(r"export const " + export_name + r": Artwork\[\]\s*=\s*(\[.*?\]);", text, re.S)
    if not match: raise ValueError("Cannot parse " + path + ": refusing unsafe duplicate import")
    return json.loads(match.group(1))

# Identify identical artworks without fetching their images.
# The existing base uses friendly slugs; new additions use aic-<museum-id>.
def catalog_id(work):
    m = re.search(r"artic\.edu/artworks/(\d+)", str(work.get("sourceUrl") or ""))
    if m: return m.group(1)
    m = re.fullmatch(r"aic-(\d+)", str(work.get("id") or ""))
    return m.group(1) if m else None

def iiif_image_key(url):
    m = re.search(r"/iiif/2/([^/]+)/", str(url or ""))
    return m.group(1) if m else None

def index_records(records):
    return {
        "ids": {str(x.get("id") or "") for x in records},
        "museum_ids": {v for x in records if (v := catalog_id(x))},
        "titles": {title_key(x.get("artist"), x.get("title")) for x in records},
        "urls": {str(x.get("sourceUrl") or "").split("?")[0].rstrip("/") for x in records},
        "image_ids": {v for x in records if (v := iiif_image_key(x.get("sourceImageUrl")))},
    }

def duplicates(a, index, file_paths=None):
    # Called BEFORE image download. Includes title variants and actual IIIF identifiers.
    item_id, museum_id = str(a["id"]), catalog_id(a)
    source = str(a.get("sourceUrl") or "").split("?")[0].rstrip("/")
    image_id = iiif_image_key(a.get("sourceImageUrl"))
    if item_id in index["ids"]: return "app-id"
    if museum_id and museum_id in index["museum_ids"]: return "museum-id"
    if source in index["urls"]: return "source-url"
    if title_key(a.get("artist"), a.get("title")) in index["titles"]: return "artist+title"
    if image_id and image_id in index["image_ids"]: return "IIIF-image-id"
    if file_paths and ("public/artworks/" + item_id + ".webp" in file_paths
                       or "public/artworks/" + item_id + "-thumb.webp" in file_paths):
        return "GitHub-image-file"
    return None

def git_tree_paths(revision):
    commit_data = gh("GET", "/git/commits/" + revision)
    sha = commit_data["tree"]["sha"]
    data = gh("GET", "/git/trees/" + sha + "?recursive=1")
    if data.get("truncated"):
        raise RuntimeError("GitHub tree listing truncated; refusing to risk duplicate image downloads.")
    return {t["path"] for t in data.get("tree", []) if t.get("type") == "blob"}

BASE_FILE = "src/games/art-slide/data/artworks.ts"
EXTRA_FILE = "src/games/art-slide/data/additional-artworks.ts"
RESEARCH_FILE = "src/games/art-slide/data/artworks-research.json"

# Fetch latest on GitHub at execution time, so results from a notebook
# that's running RIGHT NOW will be excluded when this notebook runs next.
initial_ref = gh("GET", "/git/ref/heads/" + BRANCH)["object"]["sha"]
base = parse_list(BASE_FILE, "ARTWORKS", revision=initial_ref)
old_extra = parse_list(EXTRA_FILE, "NEW_ARTWORKS", optional=True, revision=initial_ref)
published = index_records(base + old_extra)
published_files = git_tree_paths(initial_ref)
print("Already published:", len(base), "original +", len(old_extra), "additions.")
print("Git revision checked:", initial_ref[:12])
print("The notebook checks these records before downloading ANY images.")

works, research, images = [], [], {}
skip_counts = {}
def skipped(reason):
    skip_counts[reason] = skip_counts.get(reason, 0) + 1

def add(work, full, thumb, meta):
    works.append(work)
    # Register immediately, preventing duplicate downloads across search terms.
    for k, values in index_records([work]).items(): published[k].update(values)
    images["public/artworks/" + work["id"] + ".webp"] = full
    images["public/artworks/" + work["id"] + "-thumb.webp"] = thumb
    research.append(meta)
    print(f"Selected {len(works)}/{TARGET_AIC}: {work['title'][:62]}")

# Each search term has its own small paginated window. Museum API advises
# not to paginate beyond 1,000 results for a single search query.
aic_count = 0
fields = ",".join(["id", "title", "artist_title", "date_display", "date_start",
    "date_end", "image_id", "is_public_domain", "artwork_type_title",
    "classification_title", "medium_display", "style_title",
    "place_of_origin", "subject_titles", "description", "short_description"])

for term in AIC_SEARCH_TERMS:
    if aic_count >= TARGET_AIC: break
    term_count = 0
    for page in range(1, AIC_PAGES_PER_QUERY + 1):
        if aic_count >= TARGET_AIC or term_count >= AIC_PER_QUERY_QUOTA: break
        result = json_api(AIC_API + "/artworks/search", {
            "q": term,
            "query[term][is_public_domain]": "true",
            "fields": fields, "page": page, "limit": AIC_RESULTS_PER_PAGE,
        })
        items = result.get("data") or []
        if not items: break
        # Prefer museum entries with substantial curator-written explanations.
        items.sort(key=lambda a: len(plain_text(a.get("description") or a.get("short_description"))), reverse=True)
        for a in items:
            if aic_count >= TARGET_AIC or term_count >= AIC_PER_QUERY_QUOTA: break
            ident = a.get("id")
            if not ident or a.get("is_public_domain") is not True or not a.get("image_id"):
                skipped("not-public-domain-or-no-image")
                continue
            title, artist = str(a.get("title") or "").strip(), str(a.get("artist_title") or "").strip()
            if not title or not artist: continue
            medium = str(a.get("medium_display") or "")
            classification = normalized(str(a.get("artwork_type_title") or a.get("classification_title") or "") + " " + medium)
            if not re.search(r"painting|print|oil|watercolor|tempera|woodblock|etching|engraving|lithograph", classification): continue
            start = year_int(a.get("date_start")) or year_int(a.get("date_display"))
            if not start or start < MIN_YEAR or start > MAX_YEAR: continue
            end = year_int(a.get("date_end")) or start
            year = str(a.get("date_display") or start)
            image_id = str(a["image_id"])
            work_id = "aic-" + str(ident)
            url = "https://www.artic.edu/artworks/" + str(ident)
            source_image = "https://www.artic.edu/iiif/2/" + image_id + "/full/1200,/0/default.jpg"
            candidate = {"id": work_id, "artist": artist, "title": title,
                         "sourceUrl": url, "sourceImageUrl": source_image}
            reason = duplicates(candidate, published, published_files)
            if reason:
                skipped(reason)  # no image request
                continue
            curator = plain_text(a.get("description") or a.get("short_description"))
            was_shortened = len(curator) > MAX_DESCRIPTION_CHARS
            if was_shortened:
                curator = curator[:MAX_DESCRIPTION_CHARS].rsplit(" ", 1)[0].rstrip(" ,;") + "…"
            subjects = [str(v) for v in (a.get("subject_titles") or []) if v][:8]
            try:
                full, thumb, dims = download_webps(source_image)
            except (OSError, ValueError, requests.RequestException) as exc:
                skipped("image-fetch-failed-" + type(exc).__name__)
                continue
            description = curator or summary(title, artist, year, medium, str(a.get("style_title") or ""), subjects)
            work = record(work_id, title, artist, year, start, end, medium,
                "Art Institute of Chicago", url, source_image, dims,
                theme(title, medium, " ".join(subjects)), description, url if curator else None)
            if curator:
                work["descriptionModifications"] = "HTML stripped, whitespace normalized" + (", text shortened" if was_shortened else "") + "."
            meta = {
                "id": work_id, "museumId": str(ident), "sourceUrl": url,
                "descriptionKind": work["descriptionKind"],
                "descriptionLicense": "CC BY 4.0" if curator else "metadata-derived original summary",
                "descriptionLicenseUrl": CC_BY if curator else None,
                "descriptionAttribution": "Art Institute of Chicago" if curator else None,
                "descriptionModifications": work.get("descriptionModifications"),
                "style": a.get("style_title"), "place": a.get("place_of_origin"), "subjects": subjects,
            }
            add(work, full, thumb, meta)
            aic_count += 1
            term_count += 1
        print("Search", repr(term), f"page {page}, accepted {term_count}, total {aic_count}/{TARGET_AIC}")
    print("Done with", repr(term), "—", term_count, "new works")

print("Total new works selected:", len(works), "out of target", TARGET_AIC)
print("Existing/duplicate/invalid items skipped:", skip_counts)

if not works:
    print("Nothing to add. The repository already contains these paintings, or none passed validation.")
else:
    meta = gh("GET", "")
    if meta.get("full_name") != REPO: raise RuntimeError("Wrong GitHub repository!")
    perms = meta.get("permissions") or {}
    if perms and not (perms.get("push") or perms.get("admin")):
        raise RuntimeError("GITHUB_TOKEN must have Contents: Read and write.")

    # GitHub blobs are reusable if another commit arrives during the upload.
    uploaded_blobs = {}
    did_commit = False
    for attempt in range(4):
        # IMPORTANT: all reads here use the SAME parent revision. This keeps
        # the merger consistent when the previously running Colab job finishes.
        parent = gh("GET", "/git/ref/heads/" + BRANCH)["object"]["sha"]
        latest_base = parse_list(BASE_FILE, "ARTWORKS", revision=parent)
        latest_extra = parse_list(EXTRA_FILE, "NEW_ARTWORKS", optional=True, revision=parent)
        latest_index = index_records(latest_base + latest_extra)
        latest_paths = git_tree_paths(parent)
        chosen = [w for w in works if not duplicates(w, latest_index, latest_paths)]
        if not chosen:
            print("All selected paintings are already on GitHub. No new commit is necessary.")
            did_commit = True
            break
        print("Rechecked repository at", parent[:12], "—", len(chosen), "still new")
        selected_ids = {w["id"] for w in chosen}
        merged = latest_extra + chosen
        previous_research_text = repo_text(RESEARCH_FILE, optional=True, revision=parent)
        previous_research = json.loads(previous_research_text) if previous_research_text else []
        merged_research = {x["id"]: x for x in previous_research if isinstance(x, dict) and x.get("id")}
        merged_research.update({x["id"]: x for x in research if x["id"] in selected_ids})
        manifest = ("import type { Artwork } from '../types';\n\n"
                    "// Auto-generated by notebooks/ART_SLIDE_Chicago_200.ipynb.\n"
                    "export const NEW_ARTWORKS: Artwork[] = "
                    + json.dumps(merged, ensure_ascii=False, indent=2) + ";\n")
        research_text = json.dumps(list(merged_research.values()), ensure_ascii=False, indent=2) + "\n"
        tree_entries = []
        for idx, path in enumerate(sorted(images), 1):
            if path.split("/")[-1].split(".webp")[0].removesuffix("-thumb") not in selected_ids:
                continue
            # Should not occur because of duplicate checks; don't overwrite a loose file.
            if path in latest_paths:
                raise RuntimeError("Image path already on GitHub but absent from artwork metadata: " + path)
            if path not in uploaded_blobs:
                blob = images[path]
                blob_sha = gh("POST", "/git/blobs", {
                    "content": base64.b64encode(blob).decode("ascii"), "encoding": "base64"
                })["sha"]
                uploaded_blobs[path] = blob_sha
                if len(uploaded_blobs) % 25 == 0:
                    print("Uploaded", len(uploaded_blobs), "image blobs to GitHub")
                time.sleep(.15)
            tree_entries.append({"path": path, "mode": "100644", "type": "blob",
                                 "sha": uploaded_blobs[path]})
        for path, contents in {
            EXTRA_FILE: manifest,
            RESEARCH_FILE: research_text,
        }.items():
            tree_entries.append({"path": path, "mode": "100644",
                                 "type": "blob", "content": contents})
        parent_tree = gh("GET", "/git/commits/" + parent)["tree"]["sha"]
        new_tree = gh("POST", "/git/trees", {
            "base_tree": parent_tree, "tree": tree_entries
        })["sha"]
        commit = gh("POST", "/git/commits", {
            "message": "Add " + str(len(chosen)) + " new Chicago museum artworks",
            "tree": new_tree, "parents": [parent],
        })["sha"]
        try:
            gh("PATCH", "/git/refs/heads/" + BRANCH,
               {"sha": commit, "force": False})
            print("Committed", len(chosen), "new works (Chicago only).")
            print("https://github.com/" + REPO + "/commit/" + commit)
            print("Base artworks and frozen daily pool were not modified.")
            did_commit = True
            if DOWNLOAD_ZIP_BACKUP:
                with zipfile.ZipFile("art_slide_chicago_200.zip", "w", zipfile.ZIP_DEFLATED) as z:
                    for path, blob in images.items():
                        if path.split("/")[-1].split(".webp")[0].removesuffix("-thumb") in selected_ids:
                            z.writestr(path, blob)
                    z.writestr(EXTRA_FILE, manifest)
                    z.writestr(RESEARCH_FILE, research_text)
                files.download("art_slide_chicago_200.zip")
            break
        except RuntimeError as e:
            if ("HTTP 422" in str(e) or "HTTP 409" in str(e)) and attempt < 3:
                print("Main branch changed during upload. Refreshing latest data, retrying safely.")
                time.sleep(2)
                continue
            raise

    if not did_commit:
        raise RuntimeError("Could not fast-forward main; rerun to safely retry.")


## 実行後の動作

GitHubへの追加コミットが成功し、連携デプロイが完了すると、追加された作品はART SLIDEのギャラリーとランダムプレイに利用されます。途中経過を保存した作品や日替わりの既存問題は変更しません。

収集先は**シカゴ美術館のみ**です。The Metにはアクセスしません。取得した画像と解説は公開条件を確認してください。英文の詳しい解説がない作品は、公式メタデータから短い紹介文を作成します。既存の7言語のUI翻訳と、作品ごとの詳しい翻訳は別です。